# Machine Learning for Applied Research

## 3: Foundations of Supervised Learning
In this notebook, we will cover:
- What supervised learning is, and how it relates to AI, ML, and deep learning more broadly
- Regression vs. classification
- Why and how we split data into training and test sets
- The bias-variance trade-off, explained and visualized


### Introduction

Before we can actually train anything, we need to be precise about what "supervised learning" means, how it relates to the other terms you have probably heard - AI, ML, deep learning - and how to tell honestly whether a trained model is any good. This notebook builds those foundations, still using the Building Energy Efficiency dataset.

### Where Does Supervised Learning Fit?

These terms get used loosely, so it is worth being precise:

<img src="https://drive.google.com/uc?id=1mICminLcTnTkNoxGoC2TZjB9mSZhJT3H" style = "height:350px"> <br>

- **Artificial Intelligence (AI)**: the broad field of building systems that perform tasks normally requiring human intelligence: reasoning, perception, decision-making. This includes machine learning, but also rule-based systems, search algorithms, and much more that involves no learning from data at all.
- **Machine Learning (ML)**: the part of AI where a system learns patterns from data, rather than being explicitly programmed with rules. Most of what gets called "AI" in current practice is machine learning specifically.
- Within ML, three broad paradigms describe what kind of data and feedback are available:

| Paradigm | Data available | Goal | Example |
|---|---|---|---|
| Supervised learning | Labeled examples (input paired with a known correct output) | Learn a mapping from input to output | Predicting heating load from building design - this course's focus |
| Unsupervised learning | Unlabeled examples (input only) | Discover structure, patterns, or groupings | Grouping satellite images into similar land-cover types with no predefined labels |
| Reinforcement learning | An environment and a reward signal, not a fixed dataset | Learn a sequence of actions that maximizes reward over time | A robot learning which grasp motion works best through repeated trial and error |

**Where does deep learning fit in?** Deep learning is not a fourth paradigm! It is a family of models (neural networks with many layers) that can be trained in a supervised, unsupervised, or reinforcement setting. What makes deep learning distinctive is less the learning paradigm and more the type of data it tends to need: very large datasets and complex, unstructured inputs like images, audio, or free text play to its strengths. Applied engineering research often instead works with smaller, structured (tabular) datasets where classical methods like the ones in this seminar (linear models, trees, ensembles) are frequently just as effective, and considerably easier to train, tune, and interpret with limited data. Deep learning is a different tool for a different kind of problem, not a strict upgrade over what we are covering here.

We focus on supervised learning in this seminar because it is the most common starting point in applied research.

### Regression vs. Classification

Within supervised learning, the nature of the target variable determines the task type:
- **Regression**: the target is a continuous, numeric quantity (heating load, cycles to failure, gas concentration in ppm)
- **Classification**: the target is a discrete category (pass/fail, defect type, machine failure yes/no)

The same underlying features can sometimes support either framing - recall Notebook 1's predictive-maintenance scenario, where the choice depended on the actual decision being made, not just on which data happened to be available.

<img src="https://drive.google.com/uc?id=1H9OZnmgz1CNb_aRiKkq0TPRW1WXx0SPb" style = "height:350px"> <br>


#### 1.1 Exercises:
> For each scenario below, decide whether it is (i) supervised regression, (ii) supervised classification, (iii) unsupervised, or (iv) reinforcement learning:
>
> (a) Grouping satellite images into similar land-cover types, with no predefined labels.
>
> (b) Predicting the number of cycles until a material fails, from measured strain and temperature.
>
> (c) A warehouse robot learning which sequence of movements picks an item fastest, through repeated trial and reward.
>
> (d) Classifying incoming sensor readings as "normal" or "requires inspection."
>
> (e) Predicting hourly bike rental counts from weather data.

In [ ]:
# Your solution:

# a =
# b =
# c =
# d = 
# e =

##### Solution:

In [ ]:
# (a) Unsupervised: there are no labels telling us the "correct"
#     land-cover type; the goal is to discover groupings in the data
#     itself.

# (b) Supervised regression: cycles to failure is a continuous, ordered
#     quantity.

# (c) Reinforcement learning: the robot has no labeled examples of
#     "correct" motions, only a reward signal from trial and error.

# (d) Supervised classification: the target is a discrete category
#     ("normal" vs. "requires inspection"), learned from labeled examples.

# (e) Supervised regression: rental count is a continuous, numeric
#     quantity predicted from labeled historical examples.

### Splitting Data: Train, Validation, and Test Sets
A model that is evaluated on the same data it was trained on will almost always look better than it actually is because it has had the chance to memorize quirks specific to that data, rather than learning something that generalizes to new examples. This is why we split data into separate subsets before training anything:

- **Training set**: used to fit the model's parameters
- **Test set**: held out and touched only once, at the end, to estimate how the model will perform on genuinely new data
- **Validation set**: a third split, used during model development to make choices (which model, which settings) without touching the test set. We will introduce this formally later when we cover cross-validation and hyperparameter tuning; for now, a simple train/test split is enough.

In [ ]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd

energy = fetch_ucirepo(id=242)
X = energy.data.features
y = energy.data.targets

X.columns = ["Relative Compactness",
             "Surface Area",
             "Wall Area",
             "Roof Area",
             "Overall Height",
             "Orientation",
             "Glazing Area",
             "Glazing Area Distribution"]

y.columns = ["Heating Load", "Cooling Load"]

target_col = y.columns[0]   # Heating Load
df = pd.concat([X, y[[target_col]]], axis=1)

print(f"Shape: {df.shape}")
df.head()

#### 1.2 Exercises:
> (a) Import `train_test_split` from `sklearn.model_selection`.
>
> (b) This function gives us a tuple with 4 values. Split `df` into `X_train`, `X_test`, `y_train`, `y_test`, using all 8 design columns as features and `Heating Load` as the target, with `test_size=0.25` and `random_state=42`.
>
> (c) Print the number of rows in each of the four resulting objects, and confirm the split is roughly 75/25 of the original 768 rows.

In [ ]:
# Your solution:




##### Solution:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[target_col]), df[target_col], test_size=0.25, random_state=42
)

print(f"X_train: {X_train.shape[0]} rows")
print(f"X_test:  {X_test.shape[0]} rows")
print(f"y_train: {y_train.shape[0]} rows")
print(f"y_test:  {y_test.shape[0]} rows")
# 576 train / 192 test - a 75/25 split of the original 768 rows, as expected.

### The Bias-Variance Trade-off

Before going further, we need a way to measure how wrong a model's predictions are. We will use RMSE (root mean squared error) throughout this course. This metric is the typical size of a prediction error, expressed in the same units as the target itself.

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    """Root mean squared error - typical prediction error size, in the target's units."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

With training and test data in hand, we can now ask: how complex should a model be? This is where the *bias-variance trade-off* comes in:

- **Bias**: a consistent, systematic error from a model that is too simple to capture the true relationship in the data. A model with high bias underfits.
- **Variance**: error from a model that is too sensitive to the specific training data it happened to see, including its noise, so the predictions become unstable and don't generalize. A model with high variance overfits.

A common simplified way to write this: total expected error ≈ bias² + variance + irreducible noise. You generally cannot drive both bias and variance to zero at once - reducing one tends to increase the other, which is what makes model complexity a real design choice rather than "more complex is always better."

<img src="https://drive.google.com/uc?id=1sR6cxpLpAQK6R8bCA0bQCIk7e9NBfl2H" style = "height:350px"> <br>

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Revisit Overall Height from Notebook 1, this time to see how model
# complexity affects the fit
X_train_1d = X_train[["Overall Height"]]
X_test_1d = X_test[["Overall Height"]]

degrees = [1, 3, 12]
labels = ["Degree 1 (too simple)", "Degree 3 (reasonable)", "Degree 12 (too complex)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_plot = np.linspace(X_train_1d.min().values[0], X_train_1d.max().values[0], 200).reshape(-1, 1)

for ax, degree, label in zip(axes, degrees, labels):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train_1d, y_train)

    train_err = rmse(y_train, model.predict(X_train_1d))
    test_err = rmse(y_test, model.predict(X_test_1d))

    ax.scatter(X_train_1d, y_train, s=20, label="Train")
    ax.scatter(X_test_1d, y_test, s=20, label="Test", marker="x")
    ax.plot(x_plot, model.predict(x_plot), color="black")
    ax.set_title(f"{label}\nTrain RMSE: {train_err:.2f} | Test RMSE: {test_err:.2f}")
    ax.set_xlabel("Overall Height (m)")
    ax.set_ylabel("Heating Load")
    ax.legend()

plt.tight_layout()
plt.show()

Which of the three models above has high bias, and which has high variance? How can you tell from the train/test RMSE numbers alone, without even looking at the plot?

#### 1.3 Exercises:
> (a) Using all 8 features (`X_train`, `X_test`), fit a plain `LinearRegression` (equivalent to polynomial degree 1) and report its train and test RMSE.
>
> (b) Generate degree-3 polynomial features on all 8 features using `PolynomialFeatures`, refit, and compare train/test RMSE to (a).
>
> (c) Based on the train/test gap, would you describe the degree-3 model as suffering more from bias or from variance? Justify your answer in a comment.

In [ ]:
# Your solution:




##### Solution:


In [ ]:
# (a) Plain linear regression
linreg = LinearRegression()
linreg.fit(X_train, y_train)
print(f"Degree 1 (linear) - Train RMSE: {rmse(y_train, linreg.predict(X_train)):.2f}")
print(f"Degree 1 (linear) - Test RMSE:  {rmse(y_test, linreg.predict(X_test)):.2f}")

# (b) Degree-3 polynomial features on all 8 features
poly_model = make_pipeline(PolynomialFeatures(3), LinearRegression())
poly_model.fit(X_train, y_train)
print(f"\nDegree 3 (all features) - Train RMSE: {rmse(y_train, poly_model.predict(X_train)):.2f}")
print(f"Degree 3 (all features) - Test RMSE:  {rmse(y_test, poly_model.predict(X_test)):.2f}")

# (c) The degree-3 model's train RMSE drops sharply while its test RMSE
# does not improve by nearly as much (or gets worse). This is a large train/test
# gap. That pattern points to variance, not bias: the model is flexible
# enough to fit the training data very closely, including its noise, but
# that flexibility does not transfer to unseen data.

#### Challenge (optional):
> Using `X_train_1d` and `X_test_1d` (the single-feature Overall Height data from the plot above), loop over polynomial degrees 1 through 15. For each degree, fit a model and record its train and test RMSE, then plot both curves against degree on a single line chart. At which degree is test RMSE lowest?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
degrees_range = range(1, 16)
train_errors = []
test_errors = []

for degree in degrees_range:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train_1d, y_train)
    train_errors.append(rmse(y_train, model.predict(X_train_1d)))
    test_errors.append(rmse(y_test, model.predict(X_test_1d)))

plt.figure(figsize=(8, 5))
plt.plot(list(degrees_range), train_errors, marker="o", label="Train RMSE")
plt.plot(list(degrees_range), test_errors, marker="o", label="Test RMSE")
plt.xlabel("Polynomial degree")
plt.ylabel("RMSE")
plt.title("Bias-variance trade-off as model complexity increases")
plt.legend()
plt.show()

best_degree = list(degrees_range)[int(np.argmin(test_errors))]
print(f"Degree with lowest test RMSE: {best_degree}")

# Trying many settings and keeping the one with the lowest test (or,
# properly, validation) error is essentially what automated hyperparameter
# tuning does. We will cover this formally later, and do it properly with a
# validation set instead of the test set.

### Conclusion

This notebook laid the conceptual and practical groundwork for everything that follows:

- **Supervised learning in context**: one of three ML paradigms, distinct from deep learning (which is a model family, not a paradigm), and often the more appropriate choice for the smaller, structured datasets common in applied engineering research
- **Regression vs. classification**: determined by whether the target is continuous or categorical
- **Train/test splitting**: essential for honestly estimating how a model will perform on data it hasn't seen
- **The bias-variance trade-off**: the fundamental tension between a model too simple to capture the pattern, and one too flexible to generalize beyond its training data

Notebook 4 continues with model training in practice: linear models, decision trees, and ensemble methods, still on the Building Energy Efficiency dataset.
